## Recherche documentaire

Ce notebook teste `search_docs` (dans `src/tools/retrieval.py`) et compare ses trois modes : embeddings seuls, mots-clés (BM25), et hybride.

Après avoir modifié un fichier de `src/`, redémarre le noyau avant de relancer.

In [ ]:
!pip install rank_bm25 langchain-openai

In [ ]:
import os
import sys

import pandas as pd
from snowflake.snowpark.context import get_active_session

ROOT = os.path.abspath("..")
if ROOT not in sys.path:
    sys.path.append(ROOT)

from src.tools.retrieval import search_docs

session = get_active_session()
pd.set_option("display.max_colwidth", 120)

## Partie A : essais à la main

Change la question ou le mode (`dense`, `bm25`, `hybrid`) pour comparer.

In [ ]:
def show(question, **kwargs):
    result = search_docs(session, question, **kwargs)
    print("Question :", question)
    print("Mode     :", result["mode"], "| fiable :", result["reliable"], "| seuil :", result["threshold"])
    if not result["passages"]:
        print("Aucun passage")
        return None
    table = pd.DataFrame(result["passages"])[["score", "source_type", "title", "content"]]
    table["content"] = table["content"].str.replace("\n", " ").str[:110]
    return table

In [ ]:
show("The BAF09 amount is wrong and I cannot change it", mode="dense")

In [ ]:
show("The BAF09 amount is wrong and I cannot change it", mode="bm25")

In [ ]:
show("The BAF09 amount is wrong and I cannot change it", mode="hybrid")

In [ ]:
show("What is the best recipe for a chocolate cake?", mode="hybrid")

## Mesure déjà faite : le préfixe de question

Comparaison sur 60 questions de `KB_QA` (cherchées sans leur propre fiche) et 10 phrases hors sujet.

| | Sans préfixe | Avec préfixe |
|---|---|---|
| Meilleur score hors sujet | 0,597 | 0,377 |
| Score médian des vraies questions | 0,607 | 0,536 |
| Vraies questions au-dessus de tout le hors sujet | 35 / 60 | 59 / 60 |

Décision : préfixe activé, seuil de similarité à 0,45.

## Partie B : comparer les trois modes

Pour chaque question tirée de `KB_QA`, le LLM écrit une reformulation, comme si un autre utilisateur posait le même problème avec d'autres mots. On cherche avec la reformulation et on regarde si la fiche d'origine est retrouvée.

Limite : une reformulation écrite par un LLM garde souvent les mêmes mots-clés, ce qui avantage BM25. Le résultat est donc optimiste.

In [ ]:
from src.llm import make_llm

llm = make_llm()
kb = session.table("PROJECT_DB.PUBLIC.KB_QA").to_pandas()
bench = kb.sample(60, random_state=1)[["ID", "QUESTION"]].reset_index(drop=True)

INSTRUCTION = (
    "Rewrite this support ticket as another user would write it: same problem, different wording "
    "and sentence structure. Keep codes, port names and reference numbers unchanged. "
    "Reply with the rewritten ticket only, in English.\n\nTicket: "
)

bench["PARAPHRASE"] = [llm.invoke(INSTRUCTION + q).content.strip() for q in bench["QUESTION"]]
bench.head(5)

In [ ]:
def rank_of(passages, source_id):
    for rank, p in enumerate(passages, start=1):
        if p["source_id"] == source_id:
            return rank
    return None


rows, details = [], []
for mode in ("dense", "bm25", "hybrid"):
    ranks = []
    for _, r in bench.iterrows():
        res = search_docs(session, r["PARAPHRASE"], top_k=5, source_type="faq", mode=mode)
        rank = rank_of(res["passages"], r["ID"])
        ranks.append(rank)
        details.append({"mode": mode, "ID": r["ID"], "rang": rank})
    ranks = pd.Series(ranks, dtype="float")
    rows.append({
        "mode": mode,
        "questions": len(ranks),
        "fiche retrouvée en 1er": int((ranks == 1).sum()),
        "fiche dans les 5": int(ranks.notna().sum()),
    })

pd.DataFrame(rows)

### Les questions où les modes divergent

In [ ]:
wide = pd.DataFrame(details).pivot(index="ID", columns="mode", values="rang")
wide = wide.merge(bench.set_index("ID")[["PARAPHRASE"]], left_index=True, right_index=True)
wide[wide[["dense", "bm25", "hybrid"]].nunique(axis=1, dropna=False) > 1].head(15)

# Partie 3: Vérification par le LLM

In [ ]:
import importlib
import src.tools.retrieval
importlib.reload(src.tools.retrieval)
from src.tools.retrieval import search_docs, verify_passages

question = "The BAF09 amount is wrong and I cannot change it"
    #"Why can't I find the KILIMA service when creating my pricelist?"
    #"What is the best recipe for a chocolate cake?"


result = search_docs(
    session,
    question,
    top_k=5,
    mode="dense"
)

verification = verify_passages(
    llm,
    question,
    result["passages"],
    result["threshold"]
)

verification

In [ ]:
final_reliable = (
    result["reliable"]
    and verification["relevant"]
)

print("Seuil atteint       :", result["reliable"])
print("Validation LLM      :", verification["relevant"])
print("Fiabilité finale    :", final_reliable)
print("Sources pertinentes :", verification["source_ids"])
print("Raison              :", verification["reason"])

# Patie 4: Test doc_search

In [ ]:
!pip install langgraph

In [ ]:
import json
import importlib
import sys

# Recharge les modules src (le kernel garde en cache les anciennes versions).
# Deux passes pour que les modules dépendants récupèrent les fonctions rechargées.
_src_modules = [m for m in sys.modules if m == "src" or m.startswith("src.")]
for _ in range(2):
    for _name in _src_modules:
        importlib.reload(sys.modules[_name])

from src.llm import make_llm
from src.graph import build_graph
from src.trace import show_trace

llm = make_llm()

graph = build_graph(
    llm=llm,
    session=session
)
with open("../tests/scenarios.json", "r") as f:
    scenarios = json.load(f)["scenarios"]

scenario = scenarios[3] #scenario pour tester  etape 10

initial_state = {
    "ticket": scenario["ticket"],
    "fixtures": scenario.get("fixtures", {}),
    "trace": []
}

result = graph.invoke(initial_state)

print("Décision :", result["decision"])
print("Règle    :", result["decision_rule"])
print("Réponse  :", result["answer"])

show_trace(result["trace"])


In [ ]:
#Vrai doc_search
print("docs_reliable =", result["docs_reliable"])

for doc in result["docs"]:
    print(
        doc["source"],
        doc["score"],
        doc["extrait"][:200]
    )

In [ ]:
#test 7 scénarios
test_results = []

for scenario in scenarios:

    initial_state = {
        "ticket": scenario["ticket"],
        "fixtures": scenario.get("fixtures", {}),
        "trace": []
    }

    result = graph.invoke(initial_state)

    expected = scenario["expected"]

    ok_decision = result["decision"] == expected["decision"]
    ok_rule = result["decision_rule"] == expected["rule"]
    ok_gap = result["kb_gap"] == expected["kb_gap"]

    success = ok_decision and ok_rule and ok_gap

    test_results.append({
        "scenario": scenario["id"],
        "decision": result["decision"],
        "expected_decision": expected["decision"],
        "rule": result["decision_rule"],
        "expected_rule": expected["rule"],
        "kb_gap": result["kb_gap"],
        "success": success
    })

In [ ]:
df_tests = pd.DataFrame(test_results)

display(df_tests)

In [ ]:
print(
    "Scénarios conformes :",
    df_tests["success"].sum(),
    "/",
    len(df_tests)
)